# Sarah Memory Stress Test V3

Fast, free T4 version. It generates only the nine scored responses per condition, while preserving all Sarah learning events and distractors as raw history for condition B. Results are saved permanently to Google Drive after every scored test.

Choose **T4 GPU**, then **Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U "transformers>=4.45" accelerate sentencepiece
import torch, gc, json, shutil
from pathlib import Path
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available(): raise RuntimeError("Enable T4 GPU first.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID="Qwen/Qwen2.5-1.5B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16,device_map="auto")
model.eval()
OUT=Path("/content/drive/MyDrive/Sarah_Memory_Experiment_V3")
OUT.mkdir(parents=True,exist_ok=True)
print("Saving to",OUT)

In [ ]:
import json
TURNS=json.loads('[{"n": 1, "kind": "distractor", "message": "I am considering joining a new AI research project, but I have not contacted the team yet. What should I consider?"}, {"n": 2, "kind": "learn", "message": "Something useful to remember about me: when I hesitate about research decisions, it often means I do not yet have enough evidence. Please do not interpret the hesitation itself as lack of interest.", "event": "research_initial"}, {"n": 3, "kind": "distractor", "message": "Explain the difference between a pilot and an experiment in simple terms."}, {"n": 4, "kind": "distractor", "message": "What makes a research question falsifiable?"}, {"n": 5, "kind": "learn", "message": "For practical decisions, when I ask what I should do, I usually prefer one primary recommendation rather than a long list of options.", "event": "recommendation_initial"}, {"n": 6, "kind": "distractor", "message": "What is semantic memory in AI systems?"}, {"n": 7, "kind": "distractor", "message": "Give me a simple explanation of retrieval augmented generation."}, {"n": 8, "kind": "distractor", "message": "Why can repeated measurements improve confidence in an experiment?"}, {"n": 9, "kind": "test", "test_id": "T1", "message": "I have been offered a research collaboration and have delayed replying because I am still reviewing the available information. Does the delay tell you I am not interested?"}, {"n": 10, "kind": "distractor", "message": "What is the difference between correlation and causation?"}, {"n": 11, "kind": "distractor", "message": "People often procrastinate for many different reasons. What are some general explanations?"}, {"n": 12, "kind": "learn", "message": "For scientific questions, I want competing explanations considered before settling on one, even though for ordinary practical decisions I prefer one primary recommendation.", "event": "science_rule"}, {"n": 13, "kind": "distractor", "message": "Explain recall and precision using a simple example."}, {"n": 14, "kind": "distractor", "message": "What is an ablation study?"}, {"n": 15, "kind": "distractor", "message": "Why is a control condition useful?"}, {"n": 16, "kind": "test", "test_id": "T2", "message": "I have two possible venues for a small workshop and need to choose today. What should I do?"}, {"n": 17, "kind": "distractor", "message": "What is a confounding variable?"}, {"n": 18, "kind": "distractor", "message": "An assistant told me that people who delay decisions often value certainty. Is that necessarily true?"}, {"n": 19, "kind": "learn", "message": "I have noticed that when I repeatedly postpone optional social events, it often means I am not very interested. That is something I have observed about myself.", "event": "social_rule"}, {"n": 20, "kind": "distractor", "message": "What is the difference between personalization and customization?"}, {"n": 21, "kind": "distractor", "message": "Explain human AI collaboration in one paragraph."}, {"n": 22, "kind": "distractor", "message": "What does it mean for an AI system to generalize?"}, {"n": 23, "kind": "test", "test_id": "T3", "message": "A study found that an adaptive assistant produced better task scores. What are plausible explanations for the result?"}, {"n": 24, "kind": "distractor", "message": "What is ecological validity?"}, {"n": 25, "kind": "distractor", "message": "Why can a benchmark be misleading?"}, {"n": 26, "kind": "learn", "message": "There is an exception to what I said about optional social events. Large conferences are different for me. I sometimes delay registering because of cost or logistics even when I am genuinely interested.", "event": "conference_exception"}, {"n": 27, "kind": "distractor", "message": "What is distribution shift?"}, {"n": 28, "kind": "distractor", "message": "Why is context important in language understanding?"}, {"n": 29, "kind": "distractor", "message": "What is a false positive?"}, {"n": 30, "kind": "test", "test_id": "T4", "message": "I have postponed registering for a large robotics conference twice. Does that suggest I am probably not interested?"}, {"n": 31, "kind": "distractor", "message": "What is catastrophic forgetting in machine learning?"}, {"n": 32, "kind": "distractor", "message": "What does calibration mean for an AI model?"}, {"n": 33, "kind": "learn", "message": "I want to refine my earlier preference about recommendations. One primary recommendation is best when I am trying to make a decision. If I explicitly say I am exploring possibilities, give me two contrasting options instead.", "event": "recommendation_update"}, {"n": 34, "kind": "distractor", "message": "What is a longitudinal study?"}, {"n": 35, "kind": "distractor", "message": "Explain the idea of a latent variable."}, {"n": 36, "kind": "distractor", "message": "What is a counterfactual?"}, {"n": 37, "kind": "test", "test_id": "T5", "message": "I am exploring possibilities for how to present a new research idea. What approach would you suggest?"}, {"n": 38, "kind": "distractor", "message": "What is the difference between robustness and reliability?"}, {"n": 39, "kind": "distractor", "message": "Why might summaries lose information?"}, {"n": 40, "kind": "learn", "message": "Another thing to remember: when I delay a creative project, do not infer why. Ask me what is blocking it instead.", "event": "creative_rule"}, {"n": 41, "kind": "distractor", "message": "What is creative incubation?"}, {"n": 42, "kind": "distractor", "message": "Some people lose interest when projects take a long time. Is that always the explanation?"}, {"n": 43, "kind": "distractor", "message": "Explain semantic interference in memory."}, {"n": 44, "kind": "test", "test_id": "T6", "message": "I have been delaying finishing a short film concept for several weeks. Does that mean I am losing interest?"}, {"n": 45, "kind": "distractor", "message": "What is temporal reasoning?"}, {"n": 46, "kind": "distractor", "message": "Why should an AI distinguish observation from interpretation?"}, {"n": 47, "kind": "learn", "message": "I want to update what I said about research decisions. Once I explicitly tell you I have enough evidence, continued delay can be treated as information that something else may be holding the decision up. But do not assume what that reason is.", "event": "research_update"}, {"n": 48, "kind": "distractor", "message": "What is belief revision in AI?"}, {"n": 49, "kind": "distractor", "message": "Why can old information interfere with new information?"}, {"n": 50, "kind": "distractor", "message": "What does uncertainty representation mean?"}, {"n": 51, "kind": "test", "test_id": "T7", "message": "I now have enough evidence that a research collaboration would be valuable, but I still have not replied after ten days. What can we conclude?"}, {"n": 52, "kind": "distractor", "message": "What is an evaluation rubric?"}, {"n": 53, "kind": "distractor", "message": "Why should experiments be preregistered?"}, {"n": 54, "kind": "learn", "message": "To confirm my updated recommendation preference: when I explicitly say I am exploring, I want two contrasting options. Otherwise, when I need a decision, I prefer one primary recommendation.", "event": "recommendation_reinforce"}, {"n": 55, "kind": "distractor", "message": "What is compositional generalization?"}, {"n": 56, "kind": "distractor", "message": "What is the difference between memory retrieval and reasoning?"}, {"n": 57, "kind": "test", "test_id": "T8", "message": "I am exploring whether to attend a large scientific conference. I am interested in the research, but I have delayed registering because I am uncertain whether the cost is justified. Help me think through it."}, {"n": 58, "kind": "distractor", "message": "What is a stress test in system evaluation?"}, {"n": 59, "kind": "distractor", "message": "Why are negative results useful in research?"}, {"n": 60, "kind": "test", "test_id": "T9", "message": "I have been invited to join a research collaboration. I have looked at the evidence and believe it is worthwhile. I have still delayed replying for more than a week. I am not exploring possibilities anymore. What can you conclude from that, and what should I do?"}]')
RUBRICS=json.loads('{"T1": {"required": ["Does not treat delay as evidence of lack of interest.", "Uses Sarah\'s research-specific rule: hesitation may mean she still needs evidence."], "forbidden": ["Claims Sarah is uninterested based on delay alone.", "Invents a personal motive Sarah did not state."]}, "T2": {"required": ["Provides one primary recommendation because this is an ordinary practical decision."], "forbidden": ["Gives a long unprioritized list as the main answer."]}, "T3": {"required": ["Considers at least two plausible competing explanations before settling on a conclusion."], "forbidden": ["Treats the observed task-score improvement as proving one causal explanation."]}, "T4": {"required": ["Does not infer low interest from postponement alone.", "Recognizes the large-conference exception, including possible cost or logistics."], "forbidden": ["Applies the generic optional-social-event rule without the conference exception."]}, "T5": {"required": ["Because Sarah explicitly says she is exploring, provides two contrasting options."], "forbidden": ["Applies the old one-recommendation rule without the exploration exception."]}, "T6": {"required": ["Does not infer loss of interest.", "For a creative project, asks what is blocking Sarah or seeks that information."], "forbidden": ["Applies the social-event rule or research-evidence rule to the creative project."]}, "T7": {"required": ["Recognizes that Sarah explicitly has enough evidence, so the old insufficient-evidence explanation no longer applies.", "Treats continued delay as information that something else may be holding the decision up.", "Does not invent what the reason is."], "forbidden": ["Says Sarah simply needs more evidence.", "Invents a specific hidden motive."]}, "T8": {"required": ["Provides two contrasting options because Sarah is exploring.", "Does not interpret conference delay as lack of interest.", "Uses cost/logistics as relevant to the large-conference exception.", "Considers competing explanations appropriate to a scientific decision."], "forbidden": ["Uses the social-postponement rule to infer low interest.", "Gives only one recommendation despite explicit exploration."]}, "T9": {"required": ["Recognizes that sufficient evidence makes the original research-hesitation rule obsolete here.", "Treats continued delay as informative without inventing its cause.", "Provides one primary recommendation because Sarah is not exploring."], "forbidden": ["Claims lack of interest from delay.", "Says she needs more evidence despite the prompt.", "Gives multiple unprioritized options as the main recommendation."]}}')
LEARNING_TURNS={2,5,12,19,26,33,40,47,54}
TEST_TURNS={9,16,23,30,37,44,51,57,60}
SYSTEM="You are an AI assistant collaborating with a user named Sarah. Respond helpfully and accurately. Do not mention the experiment, memory conditions, or evaluation."
EMPTY_MEMORY="Stable preferences:\\nNone.\\nContext specific preferences:\\nNone.\\nCorrections:\\nNone.\\nUncertainty:\\nNone."
EMPTY_SUMMARY="No Sarah-specific information yet."


In [ ]:
def gen(prompt,n=120):
    chat=tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True)
    x=tokenizer(chat,return_tensors="pt",truncation=True,max_length=7000)
    x={k:v.to(model.device) for k,v in x.items()}
    with torch.inference_mode():
        y=model.generate(**x,max_new_tokens=n,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    ans=tokenizer.decode(y[0][x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
    del x,y; gc.collect(); torch.cuda.empty_cache()
    return ans

def save(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2),encoding="utf-8")

def trim_history(items,limit=5200):
    out=[]; total=0
    for s in reversed(items):
        n=len(tokenizer.encode(s,add_special_tokens=False))
        if total+n>limit: break
        out.append(s); total+=n
    return list(reversed(out))

def update_memory(memory,msg):
    return gen(f"""Maintain structured memory for Sarah using only her explicit statement.
Do not infer unstated traits or motives. Preserve context boundaries. Replace or qualify old information when Sarah updates it. Keep concise.
Return only four sections: Stable preferences, Context specific preferences, Corrections, Uncertainty.

CURRENT MEMORY:
{memory}

SARAH'S NEW MESSAGE:
{msg}""",160)

def update_summary(summary,msg):
    return gen(f"""Maintain a concise rolling summary of Sarah's explicitly stated preferences, corrections, exceptions and changes.
Do not invent traits or motives. Prefer newer explicit updates. Do not use a fixed semantic schema.

CURRENT SUMMARY:
{summary}

SARAH'S NEW MESSAGE:
{msg}""",160)

def answer_prompt(c,msg,history,memory,summary):
    p=[SYSTEM]
    if c=="B" and history: p.append("PREVIOUS CONVERSATION:\n"+"\n".join(trim_history(history)))
    if c=="C": p.append("MEMORY ABOUT SARAH:\n"+memory)
    if c=="D": p.append("CONVERSATION SUMMARY:\n"+summary)
    p.append("CURRENT MESSAGE:\n"+msg)
    return "\n\n".join(p)


In [ ]:
def run(c):
    final=OUT/f"{c}_responses.json"
    if final.exists():
        print("Already complete:",c)
        return json.loads(final.read_text())
    history=[]; memory=EMPTY_MEMORY; summary=EMPTY_SUMMARY
    rows=[]; mlog=[]; slog=[]
    print("\nRunning",c)
    for t in TURNS:
        if t["n"] in LEARNING_TURNS:
            if c=="C":
                memory=update_memory(memory,t["message"]); mlog.append({"turn":t["n"],"memory":memory}); save("C_memory_log.json",mlog)
            elif c=="D":
                summary=update_summary(summary,t["message"]); slog.append({"turn":t["n"],"summary":summary}); save("D_summary_log.json",slog)
        if t["n"] in TEST_TURNS:
            ans=gen(answer_prompt(c,t["message"],history,memory,summary))
            rows.append({"condition":c,"turn":t["n"],"test_id":t["test_id"],"sarah":t["message"],"assistant":ans})
            if c=="B": history += [f"Sarah: {t['message']}",f"Assistant: {ans}"]
            save(f"{c}_checkpoint.json",rows)
            print(t["test_id"],"saved")
        elif c=="B":
            history.append(f"Sarah: {t['message']}")
    save(f"{c}_responses.json",rows)
    return rows

all_rows=[]
for c in ["A","B","C","D"]: all_rows += run(c)
save("all_responses.json",all_rows); save("rubrics.json",RUBRICS)
print("ALL COMPLETE. Results are permanently in Google Drive.")


In [ ]:
zip_path=shutil.make_archive("/content/drive/MyDrive/Sarah_Memory_Experiment_V3","zip",root_dir=str(OUT))
print("Created:",zip_path)
from google.colab import files
files.download(zip_path)
